# Attending to different parts of the input with self-attention

## A simple self-attention mechanism without trainable weights

In [129]:
import torch

In [130]:
inputs = torch.tensor([
    [0.42, 0.15, 0.89], # Your (x^1)
    [0.55, 0.87, 0.66], # journey (x^2)
    [0.57, 0.85, 0.64], # starts (x^3)
    [0.22, 0.58, 0.33], # with (x^4)
    [0.77, 0.25, 0.10], # one (x^5)
    [0.05, 0.80, 0.55] # step (x^6)
])

In [131]:
input_query = inputs[1]
print(input_query)

tensor([0.5500, 0.8700, 0.6600])


In [132]:
vals = torch.matmul(inputs, input_query)
vals = torch.softmax(vals, dim=0)
vals = torch.reshape(vals, shape=[6, 1])
print(vals)

tensor([[0.1379],
        [0.2381],
        [0.2335],
        [0.1241],
        [0.1083],
        [0.1582]])


In [133]:
context_vec = inputs * vals
context_vec = torch.sum(context_vec, dim=0)
print(context_vec)

tensor([0.4405, 0.6519, 0.5681])


## Computing attention weights for all input tokens

In [134]:
atten_scores = torch.matmul(inputs, inputs.T)
print(atten_scores)

tensor([[0.9910, 0.9489, 0.9365, 0.4731, 0.4499, 0.6305],
        [0.9489, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9365, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4731, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4499, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6305, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [135]:
atten_weights = torch.softmax(atten_scores, dim=1)
print(atten_weights)

tensor([[0.2092, 0.2005, 0.1981, 0.1246, 0.1218, 0.1459],
        [0.1379, 0.2381, 0.2335, 0.1241, 0.1083, 0.1582],
        [0.1383, 0.2371, 0.2328, 0.1243, 0.1109, 0.1566],
        [0.1433, 0.2075, 0.2046, 0.1462, 0.1263, 0.1721],
        [0.1516, 0.1961, 0.1977, 0.1368, 0.1881, 0.1297],
        [0.1384, 0.2184, 0.2128, 0.1421, 0.0988, 0.1896]])


In [136]:
context_vecs = atten_weights @ inputs
print(context_vecs)

tensor([[0.4395, 0.5936, 0.5788],
        [0.4405, 0.6519, 0.5681],
        [0.4418, 0.6500, 0.5668],
        [0.4290, 0.6300, 0.5509],
        [0.4656, 0.5915, 0.5262],
        [0.4163, 0.6504, 0.5645]])


## Computing the attention weights step by step

In [137]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [138]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

In [139]:
query_2 = x_2 @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print(query_2.shape, keys.shape, values.shape)

torch.Size([2]) torch.Size([6, 2]) torch.Size([6, 2])


In [140]:
keys_2 = keys[1]
attn_score_22 = query_2 @ keys_2
print(attn_score_22)

tensor(1.8524, grad_fn=<DotBackward0>)


In [141]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2684, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)


In [142]:
d_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=0)
print(attn_weights_2)

tensor([0.1498, 0.2264, 0.2199, 0.1311, 0.0906, 0.1821],
       grad_fn=<SoftmaxBackward0>)


In [143]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3060, 0.8207], grad_fn=<SqueezeBackward4>)


## Implementing a compact SelfAttention class

In [144]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):

        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        h = keys.shape[1]
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / h**0.5, dim=0)
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in=3, d_out=2)
sa_v1(inputs)

tensor([[0.2691, 0.7274],
        [0.3596, 0.9711],
        [0.3547, 0.9578],
        [0.2233, 0.6043],
        [0.2072, 0.5609],
        [0.2645, 0.7149]], grad_fn=<MmBackward0>)

In [145]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        h = keys.shape[1]
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / h**0.5, dim=0)
        context_vec = attn_weights @ values

        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in=3, d_out=2)
sa_v2(inputs)

tensor([[-0.5705, -0.1120],
        [-0.5395, -0.1091],
        [-0.5396, -0.1091],
        [-0.4954, -0.1004],
        [-0.5194, -0.1040],
        [-0.4979, -0.1013]], grad_fn=<MmBackward0>)

## Applying a causal attention mask

In [146]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

h = keys.shape[1]
atten_scores = queries @ keys.T
atten_weights = torch.softmax(atten_scores / h**0.5, dim=-1)
print(atten_weights)

tensor([[0.1714, 0.1762, 0.1761, 0.1556, 0.1627, 0.1580],
        [0.1635, 0.1750, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1666, 0.1723, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [147]:
context_length = atten_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [148]:
masked_simple = atten_weights * mask_simple
print(masked_simple)

tensor([[0.1714, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1635, 0.1750, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1666, 0.1723, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


In [149]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4831, 0.5169, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3188, 0.3409, 0.3403, 0.0000, 0.0000, 0.0000],
        [0.2444, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1993, 0.2060, 0.2059, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


In [150]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = atten_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.3066,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1643, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0507, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1404, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0474, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [151]:
atten_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(atten_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4831, 0.5169, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3188, 0.3409, 0.3403, 0.0000, 0.0000, 0.0000],
        [0.2444, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1993, 0.2060, 0.2059, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


## Masking additional attention weights with dropout

In [152]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [153]:
example = torch.ones(6, 6)
print(layer(example))

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [154]:
print(layer(atten_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6806, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5091, 0.0000, 0.4936, 0.0000, 0.0000],
        [0.0000, 0.4121, 0.4117, 0.3870, 0.3907, 0.0000],
        [0.3248, 0.3418, 0.0000, 0.0000, 0.3249, 0.3364]],
       grad_fn=<MulBackward0>)


## Implementing a compact causal self_attention class

In [155]:
batch = torch.stack((inputs, inputs), dim=0)

In [156]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        atten_scores = queries @ keys.transpose(1, 2)
        atten_scores.masked_fill_(self.mask.bool(), -torch.inf)
        atten_weights = torch.softmax(atten_scores / keys.shape[-1]**0.5, dim=-1)
        atten_weights = self.dropout(atten_weights)
        context_vec = atten_weights @ values



        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
dropout = 0.0
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=dropout)
ca(batch)

tensor([[[-0.4470,  0.2227],
         [-0.5851,  0.0062],
         [-0.6285, -0.0629],
         [-0.5663, -0.0840],
         [-0.5516, -0.0979],
         [-0.5291, -0.1079]],

        [[-0.4470,  0.2227],
         [-0.5851,  0.0062],
         [-0.6285, -0.0629],
         [-0.5663, -0.0840],
         [-0.5516, -0.0979],
         [-0.5291, -0.1079]]], grad_fn=<UnsafeViewBackward0>)

## Stacking multiple single-head attention layers

In [157]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads=2, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)

context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0)
mha(batch)

tensor([[[-0.4470,  0.2227,  0.4723,  0.1040],
         [-0.5851,  0.0062,  0.5870,  0.3247],
         [-0.6285, -0.0629,  0.6188,  0.3854],
         [-0.5663, -0.0840,  0.5467,  0.3584],
         [-0.5516, -0.0979,  0.5312,  0.3424],
         [-0.5291, -0.1079,  0.5070,  0.3490]],

        [[-0.4470,  0.2227,  0.4723,  0.1040],
         [-0.5851,  0.0062,  0.5870,  0.3247],
         [-0.6285, -0.0629,  0.6188,  0.3854],
         [-0.5663, -0.0840,  0.5467,  0.3584],
         [-0.5516, -0.0979,  0.5312,  0.3424],
         [-0.5291, -0.1079,  0.5070,  0.3490]]], grad_fn=<CatBackward0>)

## Implementing multi-head attention with weight splits

In [160]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_in, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        atten_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        atten_scores.masked_fill_(mask_bool, -torch.inf)

        atten_weights = torch.softmax(atten_scores / keys.shape[-1]**0.5, dim=-1)
        atten_weights = self.dropout(atten_weights)

        context_vec = (atten_weights @ values).transpose(1, 2)
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        #context_vec = self.out_proj(context_vec)

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 4
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.3105, -0.2313,  0.4723,  0.1040],
         [-0.2295,  0.0311,  0.5739,  0.2996],
         [-0.2050,  0.1178,  0.6081,  0.3647],
         [-0.1635,  0.1330,  0.5419,  0.3497],
         [-0.1684,  0.1786,  0.5286,  0.3384],
         [-0.1403,  0.1693,  0.5032,  0.3400]],

        [[-0.3105, -0.2313,  0.4723,  0.1040],
         [-0.2295,  0.0311,  0.5739,  0.2996],
         [-0.2050,  0.1178,  0.6081,  0.3647],
         [-0.1635,  0.1330,  0.5419,  0.3497],
         [-0.1684,  0.1786,  0.5286,  0.3384],
         [-0.1403,  0.1693,  0.5032,  0.3400]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
